# Chess with self-play PPO

* **Environment** – `PPO/chess_engine.c` + `PPO/chess_env.py`: a vectorized chess engine written in C (bitboards, legal move generation validated with perft against `python-chess`), compiled on first import and called through `ctypes`. All the boards live in one numpy buffer and are stepped from a thread pool: one Python process, a few KB per game.
* **Agent** – `PPO/chess_policy.py`: residual CNN with a policy head over the 4672 AlphaZero-style moves and a value head. Only legal moves can be sampled (action mask).
* **Training** – `PPO/selfplay.py`: PPO with negamax GAE, a mix of opponents (current network / pool of past snapshots / baseline engines), random openings, bootstrapped time limits and potential-based reward shaping (material + mating technique) that is annealed to zero.
* **Play** – `PPO/chess_ui.py`: agent vs agent and agent vs human inside the notebook.

In [1]:
import os
import torch

from PPO.selfplay import SelfPlayPPO, EnginePlayer, NetPlayer, play_match, evaluate_policy, plot_history, elo_difference
from PPO.chess_ui import watch_game, HumanGame

torch.backends.cudnn.benchmark = True
os.makedirs('./logs', exist_ok=True)
log_file = './logs/chess.pth'

## Training

Run the cell again to resume from the checkpoint. The memory footprint is dominated by the network and the rollout (about 3 GB with the default sizes); `num_envs * num_steps` is the size of every PPO batch.

| parameter | meaning |
|---|---|
| `max_frames` | total half-moves played by all the environments (a few thousand per second on a laptop GPU) |
| `p_self`, `p_pool`, `p_engine` | probability that a new game is played against the current network, a past snapshot or a baseline engine (random / depth 1 / depth 2 search); `'lin_x'` decays from x to 0, so training starts with many engine games and moves to pure self-play |
| `shaping`, `mate_shaping`, `draw_penalty` | weights of the bootstrapping rewards (material, mating technique, penalty for drawing while ahead), linearly annealed to 0 |
| `max_plies` | games longer than this are cut and bootstrapped with the value of the last position |
| `open_plies` | number of random half-moves played at the start of each game (position diversity) |
| `target_kl` | an update is stopped early when the policy moves too far from the one that played the games |

In [ ]:
max_frames = int(1e8)

if os.path.exists(log_file):
    agent = SelfPlayPPO.load(log_file)
else:
    # Wider trunk helps represent sliding-piece attack tables in a single forward pass
    agent = SelfPlayPPO(channels=128, blocks=8, amp=True)

agent.learn(
    max_frames,
    # Longer rollout horizon + larger batch size for cleaner GAE value bootstrapping
    num_envs=256, num_steps=128, batch_size=2048, ppo_epochs=4,
    # Higher gamma/lam so midgame tactical blunders get connected to endgame results
    lr='lin_3e-4', clip_range=0.2, vf_coef=0.5, ent_coef=0.007, gamma=0.996, lam=0.97,
    clip_grad_norm=0.5, target_kl=0.03,
    max_plies=200, open_plies=(2, 10),
    # Stronger initial material shaping so 1-ply captures are heavily reinforced
    shaping='lin_1.5', mate_shaping='lin_0.5', draw_penalty='lin_0.5',
    # Keep constant or higher engine exposure and a larger snapshot pool to prevent self-play bubbles
    p_self=0.45, p_pool=0.25, p_engine=0.30, engine_depths=(0, 1, 1, 2), engine_noise=40,
    snapshot_every=20, pool_size=24,
    log_every=10, log_file=log_file, eval_games=32, eval_depths=(0, 1, 2),
)

In [ ]:
agent = SelfPlayPPO.load(log_file)
plot_history(agent.history, agent.time_elapsed)

## Evaluation

Matches are played in parallel from slightly randomised openings, the agent has white in half of the games. The Elo difference is the one that corresponds to the average score.

In [ ]:
n_games = 128
opponents = {
    'random': EnginePlayer(0),
    'engine depth 1': EnginePlayer(1, noise=10),
    'engine depth 2': EnginePlayer(2, noise=10),
    'engine depth 3': EnginePlayer(3, noise=10),
}
# the oldest and the newest snapshots of the opponent pool
snapshots = [agent.pool.items[uid] for uid in sorted(agent.pool.items)]
if snapshots:
    opponents['oldest snapshot (%.1fM frames)' % (snapshots[0]['frames'] / 1e6)] = NetPlayer(snapshots[0]['net'])
    opponents['newest snapshot (%.1fM frames)' % (snapshots[-1]['frames'] / 1e6)] = NetPlayer(snapshots[-1]['net'])

player = agent.player(deterministic=False)
for name, opponent in opponents.items():
    r = play_match(player, opponent, n_games=n_games, max_plies=300, seed=0)
    print('vs %-34s +%3i =%3i -%3i  score %.2f  elo %+5.0f   %s' % (
        name, r['wins'], r['draws'], r['losses'], r['score'], elo_difference(r['score']), r['reasons']))

## Agent vs agent

Any two players can be matched: the current network (`agent.player()`), a snapshot, or a baseline engine. `temperature` < 1 makes the agent greedier, `deterministic=True` always plays its most likely move.

In [ ]:
white = agent.player(temperature=0.5)
black = EnginePlayer(2, noise=10)      # or NetPlayer(snapshots[0]['net']) for an old version of the agent

game = watch_game(white, black, delay=0.4, names=('Agent', 'Engine depth 2'), max_plies=300)
print(game.pgn('Agent', 'Engine depth 2'))

The agent against a copy of itself (self-play, the kind of game it is trained on):

In [ ]:
game = watch_game(agent.player(), agent.player(), delay=0.3, names=('Agent', 'Agent'), max_plies=200)

## Agent vs human

Type your move in the box (SAN such as `e4`, `Nf3`, `exd5`, `O-O`, `e8=Q`, or UCI such as `g1f3`, `e7e8q`) and press Enter, or choose it from the list of legal moves. Pick your colour and press *New game* to restart; *Undo* takes back your last move together with the agent's reply.

In [ ]:
game = HumanGame(agent.player(temperature=0.3), human_color='white')
game